# Switch to 'exploration' schema

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS exploration;

USE CATALOG `operations-catalog`;
USE SCHEMA exploration;

SELECT current_schema() AS schema, current_catalog() AS catalog;

# Decode and flatten the binary fields

In [0]:
%sql
create or replace temp view bronze_purchase_order_lines_parsed as (
    select parse_json(try_cast(key as string)) as key,
    parse_json(try_cast(value as string)) as value,
    *
    except(key, value)
    from `operations-catalog`.supplychain.bronze_purchase_order_lines
);

In [0]:
%sql
select * from bronze_purchase_order_lines_parsed limit 5;

{"schema":{"type":"struct","fields":[{"type":"string","optional":false,"field":"POLineID"},{"type":"string","optional":true,"field":"PO_ID"},{"type":"string","optional":true,"field":"MaterialID"},{"type":"int32","optional":true,"field":"Quantity"},{"type":"double","optional":true,"field":"UnitCost"},{"type":"string","optional":true,"field":"__deleted"},{"type":"string","optional":true,"field":"__op"},{"type":"string","optional":true,"field":"__table"},{"type":"int64","optional":true,"field":"__lsn"},{"type":"int64","optional":true,"field":"__source_ts_ms"}],"optional":false,"name":"supplychain.public.purchase_order_lines.Value"},"payload":{"POLineID":"POL-23","PO_ID":"PO-8009","MaterialID":"RM03","Quantity":559,"UnitCost":8.75,"__deleted":"false","__op":"r","__table":"purchase_order_lines","__lsn":90951344,"__source_ts_ms":1789791513517}}

In [0]:
%sql
create or replace temp view flat_purchase_order_lines as 
select 
try_cast(key:payload:POLineID as string) as POLineID,
try_cast(value:payload:PO_ID as String) as PO_ID,
try_cast(value:payload:MaterialID as String) as MaterialID,
try_cast(value:payload:Quantity as bigint) as Quantity,
try_cast(value:payload:UnitCost as Decimal(10, 2)) as UnitCost,
try_cast(value:payload:__op as string) as __op,
try_cast(value:payload:__lsn as bigint) as __lsn,
try_cast(value:payload:__source_ts_ms as bigint) as __source_ts_ms,
topic as __topic,
partition as __partition,
offset as __offset,
timestamp as __timestamp
from bronze_purchase_order_lines_parsed;

In [0]:
%sql
select * from flat_purchase_order_lines limit 5;

In [0]:
%sql
-- check the unique values of lsn
select distinct __lsn from _sqldf limit 5;

# Schema inspection

In [0]:
%sql
describe flat_purchase_order_lines 

# Examine columns

### 1. POLineID

In [0]:
%sql
--check for uniqueness
select count(*) as count, POLineID
from flat_purchase_order_lines
group by POLineID
having count(*) > 1;

### 2. PO_ID

In [0]:
%sql
select PO_ID from flat_purchase_order_lines
where PO_ID not like 'PO_%'
limit 5;

In [0]:
%sql
select count(distinct PO_ID) from flat_purchase_order_lines
where PO_ID not in (select PO_ID from bronze_purchase_order_flat)

In [0]:
%sql
select count(*) as count from bronze_purchase_order_flat
where PO_ID not in (select PO_ID from flat_purchase_order_lines)

In [0]:
%sql
select * from bronze_purchase_order_flat where PO_ID like ' %';